# v9.5 Cholis Custom — Zenodo Direct

gtsrcmaps 완전 우회. 원본 Zenodo E²dΦ/dE → dN/dE 변환 → ΔE×exposure×Ω → expected counts.

## Cell 1: 환경

In [1]:
#!/usr/bin/env python3
"""
v9.5 Cholis Custom Likelihood — Zenodo Direct Loading
=====================================================
v9.4 대비 핵심 변경:
  - MapCube(v8.2 변환본) 대신 원본 Zenodo GDE map 직접 로드
  - 단위: E²·dΦ/dE [GeV/cm²/s/sr] → dN/dE = raw / (E_GeV² × 1000)
  - 에너지 적분: Σ_j (dN/dE_j × ΔE_j) for 38→14 bin grouping
  - Exposure: 우리 14-bin exposure cube (Cholis exposure 파일은 단위 불명확)
  - Resampling: 0.25° (240×240) → 0.1° (600×600) bilinear

검증 완료:
  Formula C (dN/dE × ΔE × exposure × Ω) → GDE-only ratio 0.75-1.07
  (GCE/bubble/iso가 빠진 상태에서 물리적으로 올바른 범위)
"""

'\nv9.5 Cholis Custom Likelihood — Zenodo Direct Loading\n=====================================================\nv9.4 대비 핵심 변경:\n  - MapCube(v8.2 변환본) 대신 원본 Zenodo GDE map 직접 로드\n  - 단위: E²·dΦ/dE [GeV/cm²/s/sr] → dN/dE = raw / (E_GeV² × 1000)\n  - 에너지 적분: Σ_j (dN/dE_j × ΔE_j) for 38→14 bin grouping\n  - Exposure: 우리 14-bin exposure cube (Cholis exposure 파일은 단위 불명확)\n  - Resampling: 0.25° (240×240) → 0.1° (600×600) bilinear\n\n검증 완료:\n  Formula C (dN/dE × ΔE × exposure × Ω) → GDE-only ratio 0.75-1.07\n  (GCE/bubble/iso가 빠진 상태에서 물리적으로 올바른 범위)\n'

## Cell 2: 에너지+binning

In [2]:
# Cell 1: 환경 설정
# ══════════════════════════════════════════════════════════════
import os, time
import numpy as np
from astropy.io import fits
from astropy.wcs import WCS
from scipy.interpolate import interp1d, RegularGridInterpolator
from scipy.ndimage import gaussian_filter
from math import lgamma
import emcee
import warnings
warnings.filterwarnings('ignore')

WORK_DIR  = '/home/haebarg/GCE-Chi-square-fitting/GCE_12yr_reproduce'
ZDIR      = '/home/haebarg/GCE-Chi-square-fitting/GCE_TEMPLATES_FILES_v3/GALACTIC_DIFFUSE_EMISSION_MAPS_0p25deg'
ZENODO_FIG = '/home/haebarg/GCE-Chi-square-fitting/GCE_TEMPLATES_FILES_v3/Figures_12_and_14_GCE_Spectra'
SANG_DIR  = os.path.join(WORK_DIR, 'GC_analysis_sanghwan')

# Zenodo GDE files — Model X = folder name "ch"
MODEL_FOLDER = 'ch'
PION_ZEN   = os.path.join(ZDIR, f'pi0_{MODEL_FOLDER}_Map_flux_E_50-814008_MeV_InnerGalaxy_60x60.fits')
BREMSS_ZEN = os.path.join(ZDIR, f'bremss_{MODEL_FOLDER}_Map_flux_E_50-814008_MeV_InnerGalaxy_60x60.fits')
ICS_ZEN    = os.path.join(ZDIR, f'ICS_{MODEL_FOLDER}_Map_flux_E_50-814008_MeV_InnerGalaxy_60x60.fits')

# Our data products
CCUBE       = os.path.join(WORK_DIR, 'GCE_12yr_ccube.fits')
EXPCUBE_CTR = os.path.join(WORK_DIR, 'GCE_12yr_expcube_center.fits')
GCE_TEMPLATE = os.path.join(WORK_DIR, 'GCE_template_NFW2.fits')
BUB_TEMPLATE = os.path.join(WORK_DIR, 'Fermi_Bubbles_template.fits')
ISO_SPECTRUM = os.path.join(WORK_DIR, 'isotropic_spectrum_ff.txt')
BUB_SPECTRUM = os.path.join(WORK_DIR, 'fermi_bubble_spectrum.txt')

# Masks (Sanghwan)
PSC_MASK_FILE  = os.path.join(SANG_DIR, 'Model/GC_mask_60x60_definitions_DR2.npy')
DISK_MASK_FILE = os.path.join(SANG_DIR, 'Model/GC_disk_mask_60x60_definitions.npy')
BUB_CONSTR = os.path.join(WORK_DIR, 'bubble_constraints.txt')
ISO_CONSTR = os.path.join(WORK_DIR, 'iso_constraints_full_err.txt')

# MCMC
NWALKERS = 100; NSTEPS = 1000; BURN_IN = 400

# Grid
NX, NY = 600, 600; PIX_DEG = 0.1; NX_FIT = 400; OFFSET = 100
S, E_idx = OFFSET, OFFSET + NX_FIT; N_BINS = 14

print("=== v9.5 Cholis Custom (Zenodo direct) ===")
for f in [PION_ZEN, BREMSS_ZEN, ICS_ZEN, CCUBE, EXPCUBE_CTR,
          GCE_TEMPLATE, BUB_TEMPLATE, ISO_SPECTRUM, BUB_SPECTRUM,
          PSC_MASK_FILE, DISK_MASK_FILE]:
    print(f'  {"✅" if os.path.exists(f) else "❌"} {os.path.basename(f)}')

=== v9.5 Cholis Custom (Zenodo direct) ===
  ✅ pi0_ch_Map_flux_E_50-814008_MeV_InnerGalaxy_60x60.fits
  ✅ bremss_ch_Map_flux_E_50-814008_MeV_InnerGalaxy_60x60.fits
  ✅ ICS_ch_Map_flux_E_50-814008_MeV_InnerGalaxy_60x60.fits
  ✅ GCE_12yr_ccube.fits
  ✅ GCE_12yr_expcube_center.fits
  ✅ GCE_template_NFW2.fits
  ✅ Fermi_Bubbles_template.fits
  ✅ isotropic_spectrum_ff.txt
  ✅ fermi_bubble_spectrum.txt
  ✅ GC_mask_60x60_definitions_DR2.npy
  ✅ GC_disk_mask_60x60_definitions.npy


## Cell 3: Zenodo GDE → counts (핵심)

In [3]:
# Cell 2: 에너지 정의 + 38→14 bin grouping
# ══════════════════════════════════════════════════════════════

# 38 energy nodes from Zenodo
e38 = fits.open(PION_ZEN)['ENERGIES'].data['Energy']  # MeV
print(f"\n38 energy nodes: {e38[0]:.1f} - {e38[-1]:.1f} MeV")

# 38-bin ΔE (adjacent node spacing)
dE38 = np.zeros(38)
for j in range(37):
    dE38[j] = e38[j+1] - e38[j]
dE38[37] = e38[37] - e38[36]

# 14-bin grouping (Cholis README: 38-bin indices 7-26 → 14 analysis bins)
bin_groups = [
    [7], [8], [9], [10], [11], [12], [13], [14], [15], [16], [17],
    [18, 19, 20],
    [21, 22, 23],
    [24, 25, 26],
]

# 14-bin edges & centers
BIN_EDGES_MEV = np.array([274.698, 357.014, 463.995, 603.034, 783.737,
    1018.59, 1323.82, 1720.51, 2236.07, 2906.12,
    3776.96, 4908.75, 10776.0, 23656.1, 51931.2])
E14_ctr_MeV = np.sqrt(BIN_EDGES_MEV[:-1] * BIN_EDGES_MEV[1:])
E14_ctr_GeV = E14_ctr_MeV * 1e-3
dE14_GeV = (BIN_EDGES_MEV[1:] - BIN_EDGES_MEV[:-1]) * 1e-3

# sr per pixel (position-dependent)
_wcs = WCS(fits.open(CCUBE)[0].header).dropaxis(2)
sr_pp = np.zeros([NY, NX])
for i in range(NX):
    for j in range(NY):
        _, b = _wcs.wcs_pix2world(i, j, 0)
        sr_pp[j, i] = np.radians(PIX_DEG)**2 * np.cos(np.radians(float(b)))

# Exposure (14-bin, 600×600)
exp14 = fits.open(EXPCUBE_CTR)[0].data  # [cm² s]

# Observed data
ccube_d = fits.open(CCUBE)[0].data

print(f"Exposure: shape={exp14.shape}")
print(f"CCUBE: shape={ccube_d.shape}")


38 energy nodes: 50.0 - 814008.0 MeV
Exposure: shape=(14, 600, 600)
CCUBE: shape=(14, 600, 600)


## Cell 4: GCE/Bubble/Iso

In [4]:
# Cell 3: Zenodo GDE → 14-bin expected counts (핵심)
# ══════════════════════════════════════════════════════════════
# 공식: counts = Σ_j [dN/dE_j × ΔE_j] × exposure × Ω
#   where dN/dE_j = raw_zenodo_j / (E_GeV_j² × 1000)

def resample_240_to_600(gde_240):
    """Bilinear resample 240×240 (0.25°) → 600×600 (0.1°)."""
    n_s = 240
    lat_s = (np.arange(n_s) - n_s/2 + 0.5) * 0.25
    lon_s = (np.arange(n_s) - n_s/2 + 0.5) * (-0.25)
    lat_t = (np.arange(600) - 300 + 0.5) * 0.1
    lon_t = (np.arange(600) - 300 + 0.5) * (-0.1)
    lat_g, lon_g = np.meshgrid(lat_t, lon_t, indexing='ij')
    interp = RegularGridInterpolator((lat_s, lon_s), gde_240,
                                      method='linear', bounds_error=False, fill_value=0)
    return interp(np.column_stack([lat_g.ravel(), lon_g.ravel()])).reshape(600, 600)


print("=== Zenodo GDE → 14-bin expected counts ===")
t0 = time.time()

pi0_raw   = fits.open(PION_ZEN)[0].data      # (38, 240, 240), E²dΦ/dE [GeV/cm²/s/sr]
bremss_raw = fits.open(BREMSS_ZEN)[0].data
ics_raw   = fits.open(ICS_ZEN)[0].data

# GDE flux per 14-bin: Σ_j [dN/dE_j × ΔE_j]  → [ph/cm²/s/sr]
gas_flux14 = np.zeros((N_BINS, NY, NX))   # pion + bremss
ics_flux14 = np.zeros((N_BINS, NY, NX))

for ib in range(N_BINS):
    for j in bin_groups[ib]:
        E_GeV_j = e38[j] * 1e-3
        # pion+bremss (coupled as c_gas)
        gas_raw_j = resample_240_to_600(pi0_raw[j] + bremss_raw[j])
        gas_dNdE_j = gas_raw_j / (E_GeV_j**2 * 1000)  # [ph/cm²/s/MeV/sr]
        gas_flux14[ib] += gas_dNdE_j * dE38[j]         # [ph/cm²/s/sr]

        # ICS
        ics_raw_j = resample_240_to_600(ics_raw[j])
        ics_dNdE_j = ics_raw_j / (E_GeV_j**2 * 1000)
        ics_flux14[ib] += ics_dNdE_j * dE38[j]
    if ib % 5 == 0:
        print(f"  bin {ib}/{N_BINS-1} done", flush=True)

# Expected counts: flux × exposure × Ω
gas_counts = gas_flux14 * exp14 * sr_pp[None, :, :]
ics_counts = ics_flux14 * exp14 * sr_pp[None, :, :]

del pi0_raw, bremss_raw, ics_raw, gas_flux14, ics_flux14

print(f"  GDE done ({(time.time()-t0):.1f}s)")

# Sanity check
print("\n--- GDE sanity check (40×40 region) ---")
print(f"{'bin':>3} {'obs_ccube':>12} {'gas+ics':>12} {'ratio':>8}  (expect 0.7-1.1)")
for ib in range(N_BINS):
    obs = ccube_d[ib, S:E_idx, S:E_idx].sum()
    pred = (gas_counts[ib] + ics_counts[ib])[S:E_idx, S:E_idx].sum()
    r = pred / obs if obs > 0 else 0
    print(f"  {ib:3d} {obs:12.0f} {pred:12.0f} {r:8.3f}")

=== Zenodo GDE → 14-bin expected counts ===
  bin 0/13 done
  bin 5/13 done
  bin 10/13 done
  GDE done (1.5s)

--- GDE sanity check (40×40 region) ---
bin    obs_ccube      gas+ics    ratio  (expect 0.7-1.1)
    0      1575718      1677352    1.065
    1      1407136      1403965    0.998
    2      1216262      1147762    0.944
    3      1009284       914705    0.906
    4       812095       697578    0.859
    5       640022       532123    0.831
    6       477060       388510    0.814
    7       342986       271478    0.792
    8       242071       186322    0.770
    9       167147       125425    0.750
   10       112340        84382    0.751
   11       152983       122846    0.803
   12        43534        41031    0.943
   13        14149        15120    1.069


## Cell 5: Masks+Constraints

In [5]:
# Cell 4: GCE / Bubble / Isotropic expected counts
# ══════════════════════════════════════════════════════════════

# ── GCE: NFW² template × BPL spectrum ──
print("\n=== GCE template ===")
gce_spatial = fits.open(GCE_TEMPLATE)[0].data.astype(np.float64)

PREFACTOR = 21e-11; INDEX1 = -1.42; INDEX2 = -2.63; BREAK_E = 2006.0

def bpl_dNdE(E_MeV):
    E = np.atleast_1d(E_MeV).astype(float)
    r = np.zeros_like(E)
    lo = E < BREAK_E; hi = ~lo
    r[lo] = PREFACTOR * (E[lo] / BREAK_E) ** INDEX1
    r[hi] = PREFACTOR * (E[hi] / BREAK_E) ** INDEX2
    return r

gce_counts = np.zeros((N_BINS, NY, NX))
for ib in range(N_BINS):
    e_sub = np.logspace(np.log10(BIN_EDGES_MEV[ib]), np.log10(BIN_EDGES_MEV[ib+1]), 20)
    bpl_int = np.trapz(bpl_dNdE(e_sub), e_sub)  # [ph/cm²/s/sr]
    gce_counts[ib] = gce_spatial * bpl_int * exp14[ib] * sr_pp

# ── Bubble: template × spectrum ──
print("=== Bubble ===")
bub_spatial = fits.open(BUB_TEMPLATE)[0].data.astype(np.float64)
bub_spec = np.loadtxt(BUB_SPECTRUM)
bub_interp = interp1d(bub_spec[:,0], bub_spec[:,1], kind='linear', fill_value='extrapolate')

bub_counts = np.zeros((N_BINS, NY, NX))
for ib in range(N_BINS):
    e_sub = np.logspace(np.log10(BIN_EDGES_MEV[ib]), np.log10(BIN_EDGES_MEV[ib+1]), 20)
    spec_int = np.trapz(bub_interp(e_sub), e_sub)
    bub_counts[ib] = bub_spatial * spec_int * exp14[ib] * sr_pp

# ── Isotropic: uniform × spectrum ──
print("=== Isotropic ===")
iso_spec = np.loadtxt(ISO_SPECTRUM)
iso_interp = interp1d(iso_spec[:,0], iso_spec[:,1], kind='linear', fill_value='extrapolate')

iso_counts = np.zeros((N_BINS, NY, NX))
for ib in range(N_BINS):
    e_sub = np.logspace(np.log10(BIN_EDGES_MEV[ib]), np.log10(BIN_EDGES_MEV[ib+1]), 20)
    spec_int = np.trapz(iso_interp(e_sub), e_sub)
    iso_counts[ib] = spec_int * exp14[ib] * sr_pp

print("All components ready ✅")


=== GCE template ===
=== Bubble ===
=== Isotropic ===
All components ready ✅


## Cell 6: MCMC+결과

In [6]:
# Cell 5: Masks + Constraints
# ══════════════════════════════════════════════════════════════
psc  = np.load(PSC_MASK_FILE)
disk = np.load(DISK_MASK_FILE)

bub_raw = np.loadtxt(BUB_CONSTR)
iso_raw = np.loadtxt(ISO_CONSTR)
bf = interp1d(bub_raw[:,0], bub_raw[:,1], fill_value='extrapolate', kind='quadratic')
bl = interp1d(bub_raw[:,0], bub_raw[:,2], fill_value='extrapolate', kind='quadratic')
bu = interp1d(bub_raw[:,0], bub_raw[:,3], fill_value='extrapolate', kind='quadratic')
if_ = interp1d(iso_raw[:,0], iso_raw[:,1], fill_value='extrapolate', kind='quadratic')
il_ = interp1d(iso_raw[:,0], iso_raw[:,2], fill_value='extrapolate', kind='quadratic')
iu_ = interp1d(iso_raw[:,0], iso_raw[:,3], fill_value='extrapolate', kind='quadratic')
bub_t  = bf(E14_ctr_GeV);  bub_le = bl(E14_ctr_GeV); bub_ue = bu(E14_ctr_GeV)
iso_t  = (E14_ctr_GeV**2)*if_(E14_ctr_GeV)
iso_le = (E14_ctr_GeV**2)*il_(E14_ctr_GeV)
iso_ue = (E14_ctr_GeV**2)*iu_(E14_ctr_GeV)

print(f"Mask: unmasked bin0={int((psc[0]*disk)[S:E_idx,S:E_idx].sum())}, "
      f"bin7={int((psc[7]*disk)[S:E_idx,S:E_idx].sum())}")
print("Ready for MCMC ✅")

Mask: unmasked bin0=54868, bin7=137346
Ready for MCMC ✅


In [7]:
# Cell 6: MCMC fit
# ══════════════════════════════════════════════════════════════

def log_fac(k):
    return np.array([lgamma(int(x)+1) for x in k.ravel()]).reshape(k.shape)


def fit_cholis():
    norms = np.zeros((N_BINS, 5))
    errors = np.zeros((N_BINS, 5))

    for ie in range(N_BINS):
        g   = gas_counts[ie, S:E_idx, S:E_idx]
        ic  = ics_counts[ie, S:E_idx, S:E_idx]
        gc  = gce_counts[ie, S:E_idx, S:E_idx]
        bc  = bub_counts[ie, S:E_idx, S:E_idx]
        isc = iso_counts[ie, S:E_idx, S:E_idx]
        obs = ccube_d[ie, S:E_idx, S:E_idx]
        exp_sr = (exp14[ie] * sr_pp)[S:E_idx, S:E_idx]
        fm  = psc[ie, S:E_idx, S:E_idx] * disk[S:E_idx, S:E_idx]
        mi  = (fm == 1)
        obs_m = obs[mi]
        lf  = log_fac(obs_m.astype(int))
        nm  = fm.sum()

        def log_posterior(p):
            for x in p:
                if x < 0: return -np.inf
            cg, ci, cge, cb, cis = p
            model = cg*g + ci*ic + cge*gc + cb*bc + cis*isc
            model_m = model[mi]
            if (model_m <= 0).any(): return -np.inf
            lhd = 2*(model_m - obs_m*np.log(model_m) + lf)

            # chi² (SED from non-convolved counts / exposure)
            iso_fl = np.sum(fm * isc / exp_sr) * cis / nm
            iso_sed = (E14_ctr_GeV[ie]**2) * iso_fl / dE14_GeV[ie]
            bub_fl = np.sum(fm * bc / exp_sr) * cb / nm
            bub_sed = (E14_ctr_GeV[ie]**2) * bub_fl / dE14_GeV[ie]

            if bub_sed > bub_t[ie]:
                chi_b = ((bub_sed - bub_t[ie]) / bub_ue[ie])**2
            else:
                chi_b = ((bub_sed - bub_t[ie]) / bub_le[ie])**2
            if iso_t[ie] < iso_sed:
                chi_i = ((iso_t[ie] - iso_sed) / iso_le[ie])**2
            else:
                chi_i = ((iso_t[ie] - iso_sed) / iso_ue[ie])**2

            return -0.5 * (np.sum(lhd) + chi_b + chi_i)

        init = np.vstack([
            np.random.uniform(0, 3,  NWALKERS),
            np.random.uniform(0, 3,  NWALKERS),
            np.random.uniform(0, 3,  NWALKERS),
            np.random.uniform(0, 10, NWALKERS),
            np.random.uniform(0, 10, NWALKERS),
        ]).T
        sampler = emcee.EnsembleSampler(NWALKERS, 5, log_posterior)
        sampler.run_mcmc(init, NSTEPS, progress=False)
        flat = sampler.get_chain(discard=BURN_IN, flat=True)
        logp = sampler.get_log_prob(discard=BURN_IN, flat=True)
        best = np.argmax(logp)
        norms[ie] = flat[best]
        errors[ie] = (np.percentile(flat, 84, axis=0) -
                      np.percentile(flat, 16, axis=0)) / 2
        print(f"  bin {ie:2d}/{N_BINS-1} — "
              f"c_gas={norms[ie,0]:.3f} c_ics={norms[ie,1]:.3f} "
              f"c_gce={norms[ie,2]:.3f} c_bub={norms[ie,3]:.3f} c_iso={norms[ie,4]:.3f}")
    return norms, errors


def gce_sed_disk_only(norms):
    sed = np.zeros(N_BINS)
    d_mask = disk[S:E_idx, S:E_idx]
    for ie in range(N_BINS):
        gc = gce_counts[ie, S:E_idx, S:E_idx]
        exp_sr = (exp14[ie] * sr_pp)[S:E_idx, S:E_idx]
        g_int = np.sum(d_mask * gc / exp_sr) * norms[ie, 2] / d_mask.sum()
        sed[ie] = (E14_ctr_GeV[ie]**2) * g_int / dE14_GeV[ie]
    return sed


def gce_sed_full_mask(norms):
    sed = np.zeros(N_BINS)
    for ie in range(N_BINS):
        gc = gce_counts[ie, S:E_idx, S:E_idx]
        exp_sr = (exp14[ie] * sr_pp)[S:E_idx, S:E_idx]
        fm = psc[ie, S:E_idx, S:E_idx] * disk[S:E_idx, S:E_idx]
        g_int = np.sum(fm * gc / exp_sr) * norms[ie, 2] / fm.sum()
        sed[ie] = (E14_ctr_GeV[ie]**2) * g_int / dE14_GeV[ie]
    return sed


print('=' * 85)
print(f'v9.5 Cholis Custom (Zenodo direct) — Model X')
print(f'  MCMC={NWALKERS}×{NSTEPS}, mask=Sanghwan')
print('=' * 85)

t0 = time.time()
norms_ch, errors_ch = fit_cholis()
print(f'\nFit time: {(time.time()-t0)/60:.1f} min')

sed_disk = gce_sed_disk_only(norms_ch)
sed_full = gce_sed_full_mask(norms_ch)

zen = np.loadtxt(os.path.join(ZENODO_FIG,
      'GCE_ModelX_flux_Inner40x40_masked_disk.dat'))[:N_BINS, 1]
r_disk = sed_disk / zen
r_full = sed_full / zen

print('\n' + '=' * 105)
print('RESULT: v9.5 vs Zenodo')
print('=' * 105)
print(f'{"ie":>3} {"E[GeV]":>8} {"Zenodo":>11} '
      f'{"disk_only":>11} {"r(disk)":>8} '
      f'{"full_mask":>11} {"r(full)":>8} '
      f'{"c_gas":>7} {"c_ics":>7} {"c_gce":>7} {"c_bub":>7} {"c_iso":>7}')
print('-' * 105)
for ie in range(N_BINS):
    n = norms_ch[ie]
    print(f'{ie:3d} {E14_ctr_GeV[ie]:8.3f} {zen[ie]:11.3e} '
          f'{sed_disk[ie]:11.3e} {r_disk[ie]:8.3f} '
          f'{sed_full[ie]:11.3e} {r_full[ie]:8.3f} '
          f'{n[0]:7.3f} {n[1]:7.3f} {n[2]:7.3f} {n[3]:7.3f} {n[4]:7.3f}')
print('-' * 105)
md = r_disk.mean(); mf = r_full.mean()
md_mid = r_disk[4:11].mean(); mf_mid = r_full[4:11].mean()
print(f'{"All bins avg":>20s}  {" ":>11s} {md:8.3f} {" ":>11s} {mf:8.3f}')
print(f'{"1-10 GeV avg":>20s}  {" ":>11s} {md_mid:8.3f} {" ":>11s} {mf_mid:8.3f}')

print('\n' + '=' * 85)
print('COMPARISON')
print('=' * 85)
print(f'  v8.8 (gtsrcmaps, full_mask SED):     1-10 GeV = 0.755')
print(f'  v9.2 (gtsrcmaps, disk_only SED):     1-10 GeV = 0.854')
print(f'  v9.5 (Zenodo direct, full_mask SED):  1-10 GeV = {mf_mid:.3f}')
print(f'  v9.5 (Zenodo direct, disk_only SED):  1-10 GeV = {md_mid:.3f}')
print(f'  Zenodo target:                        1-10 GeV = 1.000')

v9.5 Cholis Custom (Zenodo direct) — Model X
  MCMC=100×1000, mask=Sanghwan
  bin  0/13 — c_gas=0.297 c_ics=1.397 c_gce=0.858 c_bub=1.070 c_iso=0.001
  bin  1/13 — c_gas=0.323 c_ics=1.478 c_gce=0.313 c_bub=1.074 c_iso=0.000
  bin  2/13 — c_gas=0.342 c_ics=1.529 c_gce=1.017 c_bub=0.863 c_iso=0.001
  bin  3/13 — c_gas=0.370 c_ics=1.517 c_gce=1.453 c_bub=0.854 c_iso=0.001
  bin  4/13 — c_gas=0.394 c_ics=1.574 c_gce=1.508 c_bub=0.774 c_iso=0.000
  bin  5/13 — c_gas=0.401 c_ics=1.554 c_gce=1.615 c_bub=0.716 c_iso=0.000
  bin  6/13 — c_gas=0.413 c_ics=1.528 c_gce=1.418 c_bub=0.694 c_iso=0.000
  bin  7/13 — c_gas=0.423 c_ics=1.514 c_gce=1.220 c_bub=0.633 c_iso=0.000
  bin  8/13 — c_gas=0.449 c_ics=1.511 c_gce=1.122 c_bub=0.602 c_iso=0.000
  bin  9/13 — c_gas=0.432 c_ics=1.512 c_gce=1.365 c_bub=0.616 c_iso=0.001
  bin 10/13 — c_gas=0.429 c_ics=1.488 c_gce=1.518 c_bub=0.648 c_iso=0.000
  bin 11/13 — c_gas=0.440 c_ics=1.444 c_gce=1.424 c_bub=0.492 c_iso=0.005
  bin 12/13 — c_gas=0.360 c_ics=1.34